In [0]:
accident_path = "/Volumes/workspace/default/urbanpulse_bronze/accidents"

accident_files = dbutils.fs.ls(accident_path)

for file in accident_files:
    print(file.name, "|", file.size, "bytes")

In [0]:
from pyspark.sql import functions as F

for file in accident_files:
    print(f"\nFILE: {file.name}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .csv(file.path)
    )

    print("Rows:", df.count())
    print("Columns:", df.columns)
    df.printSchema()
    display(df.limit(5))

In [0]:
accident_dfs = {}

for file in accident_files:
    dataset_name = file.name.replace(".csv", "")

    accident_dfs[dataset_name] = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .csv(file.path)
    )

print("Loaded accident datasets:", len(accident_dfs))

for name, df in accident_dfs.items():
    print(name, "→", df.count(), "rows")

In [0]:
accident_data = {
    "crash_statistics": accident_dfs["b75c6222-4488-4d03-a20c-4f8ac1c2a1e5"],
    "crash_victims": accident_dfs["c5a50842-cba9-4217-9202-bf40d9788789"],
    "accused_vehicles_victims": accident_dfs["d62b502e-af7d-4bc3-8784-664e3e0b8353"],
    "vehicle_crash_types": accident_dfs["ebf87168-a3d4-44a0-9041-b6a386478cb9"]
}

for name, df in accident_data.items():
    print(f"{name}: {df.count()} rows")

In [0]:
from pyspark.sql import functions as F

for name, df in accident_data.items():
    print(f"\n--- {name.upper()} ---")

    print("Total rows:", df.count())
    print("Duplicate rows:", df.count() - df.dropDuplicates().count())

    null_counts = df.select([
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ])

    display(null_counts)

In [0]:
from pyspark.sql import functions as F

for name, df in accident_data.items():
    print(f"\n--- {name.upper()} ---")

    null_summary = df.select([
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ])

    display(
        null_summary.select(
            F.explode(
                F.array(*[
                    F.struct(
                        F.lit(c).alias("column"),
                        F.col(c).alias("missing_values")
                    )
                    for c in df.columns
                ])
            ).alias("result")
        ).select("result.*")
    )

In [0]:
from pyspark.sql import functions as F

clean_accident_data = {}

for name, df in accident_data.items():

    # Remove extra spaces from column names
    for old_col in df.columns:
        new_col = (
            old_col.strip()
            .lower()
            .replace("/", "_")
            .replace(",", "")
            .replace(" ", "_")
            .replace("-", "_")
        )

        df = df.withColumnRenamed(old_col, new_col)

    # Remove exact duplicate records
    df = df.dropDuplicates()

    clean_accident_data[name] = df

    print(f"{name}: {df.count()} clean records")

In [0]:
for name, df in clean_accident_data.items():

    table_name = f"workspace.default.silver_{name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Saved: {table_name} | Records: {df.count()}")